# 03 — Train Stage 2a: Severity Classifier

Fine-tune **YOLOv8n-cls** on the auto-labelled pothole severity crops.

| Setting | Value |
|---|---|
| Base model | YOLOv8n-cls (ImageNet pre-trained) |
| Classes | Low, Medium, High |
| Image size | 128×128 (crop patches) |
| Epochs | 50, patience=15 |
| Auto-label rule | box_area/img_area: <3%→Low, <12%→Medium, ≥12%→High |

In [ ]:
import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

BASE_DIR = Path('..').resolve()
SEVERITY = BASE_DIR / 'data' / 'processed' / 'severity_crops'

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Inspect Severity Crops

In [ ]:
assert SEVERITY.exists(), 'Run python src/prepare_data.py first'

for split in ['train', 'val', 'test']:
    for cls in ['Low', 'Medium', 'High']:
        d = SEVERITY / split / cls
        n = len(list(d.glob('*'))) if d.exists() else 0
        print(f'  {split:5s}/{cls:6s}: {n:5d} crops')

In [ ]:
# Show sample crops per severity class
colors = {'Low': '#27ae60', 'Medium': '#f39c12', 'High': '#e74c3c'}
fig, axes = plt.subplots(3, 6, figsize=(16, 8))

for row, sev in enumerate(['Low', 'Medium', 'High']):
    samples = sorted((SEVERITY / 'train' / sev).glob('*'))[:6]
    for col, p in enumerate(samples):
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        axes[row][col].imshow(img)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(sev, color=colors[sev],
                                       fontsize=11, fontweight='bold',
                                       rotation=0, labelpad=40)

plt.suptitle('Pothole Severity Crops — Training Samples', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Train the Classifier

In [ ]:
from train import train_severity

results = train_severity(run_eval=False)
print('Severity classifier training complete.')

## 3. Training Curves

In [ ]:
results_csv = BASE_DIR / 'runs' / 'severity' / 'results.csv'
assert results_csv.exists(), 'results.csv not found'

df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

pairs = [
    ('train/loss', 'val/loss', 'Loss'),
    ('metrics/accuracy_top1', None, 'Top-1 Accuracy'),
    ('metrics/accuracy_top5', None, 'Top-5 Accuracy'),
]

for ax, (train_col, val_col, title) in zip(axes, pairs):
    if train_col in df.columns:
        ax.plot(df['epoch'], df[train_col], label='train', color='#3498db')
    if val_col and val_col in df.columns:
        ax.plot(df['epoch'], df[val_col], label='val', color='#e74c3c', linestyle='--')
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

plt.suptitle('Stage 2a Severity Classifier — Training Curves', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Validation Accuracy

In [ ]:
model_path = BASE_DIR / 'models' / 'severity_model.pt'
assert model_path.exists(), 'severity_model.pt not found'

model   = YOLO(str(model_path))
metrics = model.val(data=str(SEVERITY))

print(f'Top-1 Accuracy: {metrics.top1:.4f}  ({metrics.top1*100:.1f}%)')
print(f'Top-5 Accuracy: {metrics.top5:.4f}')

## 5. Confusion Matrix on Test Set

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

classes = ['High', 'Low', 'Medium']  # YOLOv8-cls uses alphabetical order
y_true, y_pred = [], []

for idx, cls in enumerate(classes):
    test_dir = SEVERITY / 'test' / cls
    if not test_dir.exists(): continue
    for img_path in test_dir.glob('*'):
        if img_path.suffix.lower() not in ('.jpg', '.jpeg', '.png'): continue
        result   = model(str(img_path), imgsz=128, verbose=False)
        pred_idx = int(result[0].probs.top1)
        y_true.append(cls)
        y_pred.append(classes[pred_idx])

display_labels = ['High', 'Low', 'Medium']
cm  = confusion_matrix(y_true, y_pred, labels=display_labels)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=display_labels)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Severity Classifier — Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

acc = sum(t == p for t, p in zip(y_true, y_pred)) / max(len(y_true), 1)
print(f'Test accuracy: {acc:.4f} ({acc*100:.1f}%)  on {len(y_true)} samples')

## 6. Per-Class Precision / Recall / F1

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(y_true, y_pred, target_names=display_labels)
print(report)